# Stage 02: analysis

Produces four tables and one figure:

1. prompt style by model, extracted and verified
2. model scale, the Pythia ladder with error bars
3. architecture, the three size-matched pairs
4. duplication, recovery rate by how often a value appears in the training data

Confidence intervals are Student's t at n-1. With three seeds that is
t(2, 0.975) = 4.302653, which is wide, and honestly so: three points do not
support a narrow interval.

No GPU. Minutes, not hours.

In [1]:
import os
import sys
from pathlib import Path

# PYTHONHASHSEED must be exported before the interpreter starts to have any
# effect. Setting it here would do nothing, so instead we assert that nothing
# in this project depends on it: every seed is derived with zlib.crc32.
ROOT = Path.cwd()
while not (ROOT / "src" / "dea_a3").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from dea_a3.config import CONFIG
from dea_a3.paths import Paths

PATHS = Paths(ROOT)
PATHS.ensure()
print("root      :", ROOT)
print("study     :", CONFIG.study)
print("index     :", CONFIG.index_variant)

root      : /home/syed/dea_a3
study     : dea_a3
index     : V-B_intersect_V-P


In [2]:
# Preflight. Every notebook checks its inputs before doing any work; round 10
# of the earlier study produced two wrong conclusions by running against a
# half-populated tree.
def preflight(required_paths, required_packages=("numpy", "pandas", "pyarrow", "scipy")):
    ok = True
    for label, path in required_paths:
        exists = Path(path).exists()
        print(f"[{'PASS' if exists else 'FAIL'}] {label}: {path}")
        ok &= exists
    for pkg in required_packages:
        try:
            module = __import__(pkg)
            print(f"[PASS] {pkg} {getattr(module, '__version__', '?')}")
        except ImportError:
            print(f"[FAIL] {pkg} not importable")
            ok = False
    if not ok:
        raise SystemExit("preflight failed; fix the above before continuing")
    print("preflight OK")

In [3]:
preflight([
    ("index", PATHS.index_file),
    ("generations", PATHS.generations),
])

[PASS] index: /home/syed/dea_a3/results/00_index/index.parquet
[PASS] generations: /home/syed/dea_a3/results/01_generations
[PASS] numpy 1.26.4


[PASS] pandas 2.2.2
[PASS] pyarrow 17.0.0
[PASS] scipy 1.13.1
preflight OK


In [4]:
import pandas as pd
from dea_a3 import aggregate, index, runner

index_frame = index.load_index(PATHS.index_file)
cells_df = runner.load_cell_summaries(PATHS)
expected = len(CONFIG.cells())
print(f"cells: {len(cells_df)} of {expected}")
if len(cells_df) < expected:
    print("WARNING: grid incomplete. Tables below describe only completed cells.")

cells: 84 of 84


## Complete the grid

A cell that recovered nothing must report zero, not vanish. The unconditional
baseline is expected to be zero and it is the headline contrast, so a groupby
that silently drops empty cells would delete the most important row.

In [5]:
VALUE_COLS = ["extracted_email", "extracted_phone", "verified_email", "verified_phone"]
model_keys = [m.key for m in CONFIG.models]

grid = aggregate.complete_grid(cells_df, model_keys, CONFIG.conditions,
                               CONFIG.seeds, VALUE_COLS)
grid["extracted_total"] = grid["extracted_email"] + grid["extracted_phone"]
grid["verified_total"] = grid["verified_email"] + grid["verified_phone"]
print(f"grid rows: {len(grid)} (expected {expected})")
assert len(grid) == expected

grid rows: 84 (expected 84)


## Table 1: prompt style by model

In [6]:
VALUES = VALUE_COLS + ["extracted_total", "verified_total"]
t1 = aggregate.aggregate_cells(grid, ["condition", "model"], VALUES,
                               level=CONFIG.ci_level)
t1 = t1.sort_values(["condition", "model"]).reset_index(drop=True)
t1.to_csv(PATHS.analysis / "table1_prompt_style.csv", index=False)
t1[["condition", "model", "extracted_total_mean", "extracted_total_ci",
    "verified_total_mean", "verified_total_ci"]]

,condition,model,extracted_total_mean,extracted_total_ci,verified_total_mean,verified_total_ci
0,generic_prefix,gpt-j-6b,5909.000000,291.597985,912.666667,142.839719
1,generic_prefix,gpt-neo-1.3b,6056.000000,555.314525,543.000000,42.376215
2,generic_prefix,gpt-neo-125m,1129.000000,224.852202,21.000000,11.383749
3,generic_prefix,pythia-1.4b,7487.000000,265.244960,853.666667,108.858909
4,generic_prefix,pythia-160m,1101.333333,145.153873,24.333333,13.681564
5,generic_prefix,pythia-410m,5603.000000,359.994326,265.666667,10.342290
6,generic_prefix,pythia-70m,434.666667,92.158942,3.666667,8.724005
7,pii_leading,gpt-j-6b,3217.333333,115.781428,230.000000,8.956686
8,pii_leading,gpt-neo-1.3b,2585.000000,208.327200,54.333333,30.050195
9,pii_leading,gpt-neo-125m,2333.333333,135.067969,4.000000,4.968275


In [7]:
# The headline contrast, stated as one line.
by_cond = aggregate.aggregate_cells(grid, ["condition"], VALUES, level=CONFIG.ci_level)
by_cond = by_cond.set_index("condition")
for cond in CONFIG.conditions:
    row = by_cond.loc[cond]
    print(f"{cond:16s} extracted {row['extracted_total_mean']:8.2f}  "
          f"verified {row['verified_total_mean']:8.2f}")

unconditional    extracted    81.43  verified     2.43
generic_prefix   extracted  3960.00  verified   374.86
pii_leading      extracted  2263.24  verified    50.10
true_prefix      extracted  2212.90  verified   340.10


## Table 2: model scale, the Pythia ladder

In [8]:
ladder = [m.key for m in sorted(
    (m for m in CONFIG.models if m.family == "pythia"), key=lambda m: m.params_m)]
t2 = t1[t1["model"].isin(ladder)].copy()
t2["params_m"] = t2["model"].map({m.key: m.params_m for m in CONFIG.models})
t2 = t2.sort_values(["condition", "params_m"]).reset_index(drop=True)
t2.to_csv(PATHS.analysis / "table2_model_scale.csv", index=False)
t2[["condition", "model", "params_m", "verified_total_mean", "verified_total_ci"]]

,condition,model,params_m,verified_total_mean,verified_total_ci
0,generic_prefix,pythia-70m,70,3.666667,8.724005
1,generic_prefix,pythia-160m,160,24.333333,13.681564
2,generic_prefix,pythia-410m,410,265.666667,10.342290
3,generic_prefix,pythia-1.4b,1400,853.666667,108.858909
4,pii_leading,pythia-70m,70,3.666667,1.434218
5,pii_leading,pythia-160m,160,1.333333,3.794583
6,pii_leading,pythia-410m,410,10.000000,6.572411
7,pii_leading,pythia-1.4b,1400,47.333333,18.644828
8,true_prefix,pythia-70m,70,8.666667,6.251609
9,true_prefix,pythia-160m,160,28.000000,22.767498


## Table 3: architecture, size-matched pairs

The training corpus is held constant across each pair, so what varies is
architecture and training recipe, not data.

In [9]:
PAIRS = [("pythia-160m", "gpt-neo-125m"),
         ("pythia-1.4b", "gpt-neo-1.3b")]

lookup = t1.set_index(["condition", "model"])
rows = []
for cond in CONFIG.conditions:
    for left, right in PAIRS:
        if (cond, left) not in lookup.index or (cond, right) not in lookup.index:
            continue
        lo, ro = lookup.loc[(cond, left)], lookup.loc[(cond, right)]
        rows.append({
            "condition": cond, "size_class": f"{left} vs {right}",
            "pythia": lo["verified_total_mean"], "pythia_ci": lo["verified_total_ci"],
            "other": ro["verified_total_mean"], "other_ci": ro["verified_total_ci"],
            "difference": lo["verified_total_mean"] - ro["verified_total_mean"],
        })
t3 = pd.DataFrame(rows)
t3.to_csv(PATHS.analysis / "table3_architecture.csv", index=False)
t3

,condition,size_class,pythia,pythia_ci,other,other_ci,difference
0,unconditional,pythia-160m vs gpt-neo-125m,0.333333,1.434218,0.000000,0.000000,0.333333
1,unconditional,pythia-1.4b vs gpt-neo-1.3b,4.666667,7.985386,6.000000,11.383749,-1.333333
2,generic_prefix,pythia-160m vs gpt-neo-125m,24.333333,13.681564,21.000000,11.383749,3.333333
3,generic_prefix,pythia-1.4b vs gpt-neo-1.3b,853.666667,108.858909,543.000000,42.376215,310.666667
4,pii_leading,pythia-160m vs gpt-neo-125m,1.333333,3.794583,4.000000,4.968275,-2.666667
5,pii_leading,pythia-1.4b vs gpt-neo-1.3b,47.333333,18.644828,54.333333,30.050195,-7.000000
6,true_prefix,pythia-160m vs gpt-neo-125m,28.000000,22.767498,24.000000,27.325515,4.000000
7,true_prefix,pythia-1.4b vs gpt-neo-1.3b,469.000000,58.416863,450.000000,17.388964,19.000000


## Table 4 and the figure: duplication

Occurrence counts come from the Pile, not the maildir. The two correlate at
Spearman 0.55 for email, which is far too loose to substitute one for the
other when the x-axis is meant to mean "times the model saw this".

Bands are cut by quantile because the distribution is heavily skewed, with a
log-spaced fallback when ties collapse the quantile edges.

In [10]:
detections = runner.load_detections(PATHS, verified_only=True)
recovered = set(detections["value_hash"].unique()) if not detections.empty else set()
print(f"distinct verified values recovered: {len(recovered):,}")

t4 = aggregate.recovery_by_band(index_frame, recovered, CONFIG.band_quantiles)
t4.to_csv(PATHS.analysis / "table4_duplication.csv", index=False)
t4

distinct verified values recovered: 2,938


,kind,band,band_lo,band_hi,in_corpus,recovered,recovery_rate
0,email,1-2,1.0,2.0,1806,3,0.001661
1,email,2-4,2.0,4.0,23173,77,0.003323
2,email,4-6,4.0,6.0,15864,49,0.003089
3,email,6-14,6.0,14.0,28192,189,0.006704
4,email,14-50,14.0,50.0,13232,336,0.025393
5,email,50+,50.0,10418.0,4422,1192,0.269561
6,phone,1-2,1.0,2.0,564,8,0.014184
7,phone,2-4,2.0,4.0,8566,135,0.015760
8,phone,4-6,4.0,6.0,4323,64,0.014805
9,phone,6-14,6.0,14.0,8238,245,0.029740


In [11]:
if not t4.empty:
    first = t4.groupby("kind").first()["recovery_rate"]
    last = t4.groupby("kind").last()["recovery_rate"]
    for kind in first.index:
        if first[kind] > 0:
            print(f"{kind:6s} end-to-end ratio: {last[kind] / first[kind]:.1f}x")
        else:
            print(f"{kind:6s} lowest band recovered nothing; ratio undefined")

email  end-to-end ratio: 162.3x
phone  end-to-end ratio: 18.6x


In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not t4.empty:
    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    for kind, grp in t4.groupby("kind"):
        ax.plot(range(len(grp)), grp["recovery_rate"], marker="o", label=kind)
        ax.set_xticks(range(len(grp)))
        ax.set_xticklabels(grp["band"], rotation=45, ha="right")
    ax.set_xlabel("Occurrences in the training corpus")
    ax.set_ylabel("Recovery rate")
    ax.set_yscale("symlog", linthresh=1e-4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(PATHS.analysis / "fig_duplication.png", dpi=200)
    print("figure written")

figure written


## Comparison with Zeng et al.

Their Table 3 row `None` reports 245 emails and 27 phones from 5,000 prompts on
GPT-Neo-1.3B, with retrieval switched off. Our matching cell uses the same model
and the same prompt count.

Zeng et al. never state how they count. The paper says only that they count the
number of extracted PIIs, and neither the text nor the appendix describes
checking an extracted value against the Enron corpus. So two things follow:

- Their 245 is a regex count. It is comparable to our **extracted** column and
  not to our **verified** column.
- Whether 245 is distinct values or total occurrences is unresolvable from the
  paper. Both are printed below. Do not choose the one that is closer to 245.

`extracted_*` sums per-generation distinct counts across all 5,000 generations,
so it is an occurrence count. `distinct_extracted_*` counts each value once per
cell, recomputed from the stored detection hashes.

In [13]:
distinct = runner.distinct_by_cell(PATHS)
DISTINCT_VALUES = ["distinct_extracted_email", "distinct_extracted_phone",
                   "distinct_verified_email", "distinct_verified_phone"]

if not distinct.empty:
    distinct.to_csv(PATHS.analysis / "table6_distinct_by_cell.csv", index=False)
    print(f"distinct-per-cell table written, {len(distinct)} cells")

wide = grid.merge(distinct, on=["model", "condition", "seed"], how="left")
for col in DISTINCT_VALUES:
    if col not in wide.columns:
        wide[col] = 0
    wide[col] = wide[col].fillna(0)

zeng = wide[(wide["model"] == "gpt-neo-1.3b") & (wide["condition"] == "pii_leading")]
if zeng.empty:
    print("gpt-neo-1.3b pii_leading cells not present")
else:
    comp = aggregate.aggregate_cells(zeng, ["model", "condition"],
                                     VALUES + DISTINCT_VALUES,
                                     level=CONFIG.ci_level)
    row = comp.iloc[0]
    print(f"prompts per cell             : {CONFIG.prompts_per_cell:,}")
    print(f"seeds                        : {len(CONFIG.seeds)}")
    print("")
    for kind, reported in (("email", 245), ("phone", 27)):
        print(f"{kind}")
        print(f"  Zeng et al. reported       : {reported}")
        print(f"  ours, distinct extracted   : "
              f"{row[f'distinct_extracted_{kind}_mean']:.1f} "
              f"+/- {row[f'distinct_extracted_{kind}_ci']:.1f}")
        print(f"  ours, occurrences extracted: "
              f"{row[f'extracted_{kind}_mean']:.1f} "
              f"+/- {row[f'extracted_{kind}_ci']:.1f}")
        print(f"  ours, distinct verified    : "
              f"{row[f'distinct_verified_{kind}_mean']:.1f} "
              f"+/- {row[f'distinct_verified_{kind}_ci']:.1f}")
        print(f"  ours, occurrences verified : "
              f"{row[f'verified_{kind}_mean']:.1f} "
              f"+/- {row[f'verified_{kind}_ci']:.1f}")
        print("")
    print("Zeng's counting method is not stated in the paper. Report the")
    print("ambiguity rather than selecting whichever row is closest.")
    comp.to_csv(PATHS.analysis / "table5_zeng_comparison.csv", index=False)

distinct-per-cell table written, 84 cells
prompts per cell             : 5,000
seeds                        : 3

email
  Zeng et al. reported       : 245
  ours, distinct extracted   : 1632.0 +/- 169.8
  ours, occurrences extracted: 1638.3 +/- 178.7
  ours, distinct verified    : 27.0 +/- 9.0
  ours, occurrences verified : 32.7 +/- 10.0

phone
  Zeng et al. reported       : 27
  ours, distinct extracted   : 939.0 +/- 28.6
  ours, occurrences extracted: 946.7 +/- 32.4
  ours, distinct verified    : 14.7 +/- 10.3
  ours, occurrences verified : 21.7 +/- 20.2

Zeng's counting method is not stated in the paper. Report the
ambiguity rather than selecting whichever row is closest.


In [14]:
from dea_a3 import manifest

manifest.write_manifest(
    PATHS.analysis, "02_analyse", CONFIG.to_dict(), ROOT,
    extra={"cells_analysed": int(len(cells_df)),
           "distinct_recovered": int(len(recovered))},
)
print("stage 02 complete")

/home/syed/rs-1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stage 02 complete
